# Step 1: Clean Raw List

Parses the downloaded HTML search-result pages from the Bundesverfassungsgericht website,
extracts the URL and label of each decision, removes duplicates, derives a document ID,
and saves the cleaned list to CSV.

**Input:** `data/1_list_raw/*.html` – one HTML file per search-result page (50 items each)  
**Output:** `data/2_list_clean/clean_list.csv` – columns: `link`, `label`, `id`

In [1]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# Parse each search-result HTML page and collect (url, label) pairs.
# Each page must contain exactly 50 result items – raises ValueError if not.
results = []
for f in os.listdir("data/1_list_raw/"):
    if f[-5:] == ".html":
        with open("data/1_list_raw/" + f, "r", encoding="utf-8") as file:
            soup = BeautifulSoup(file, "html.parser")    
        items = soup.find_all("li", class_="l-search-wrapper__result-item")
        if len(items) != 50:
            raise ValueError
        for item in items:
            link_tag = item.find("a", class_="c-teaser-search-result__link")
            if link_tag and link_tag.get("href"):
                url = link_tag["href"]
                text = link_tag.get_text(strip=True)
                results.append((url, text))
    

In [ ]:
# Convert the list of (url, label) tuples into a DataFrame
data = pd.DataFrame(results, columns=['link', 'label'])
data.head()

In [4]:
# remove duplicates
data = data[~data.duplicated()]

In [ ]:
# Extract document ID from URL: last path segment without '.html'
# e.g. '.../rk20241205_1bvr240624.html' -> 'rk20241205_1bvr240624'
data['id'] = data.link.apply(lambda url: re.search(r'/([^/]+)\.html$', url).group(1))

In [ ]:
# Save the cleaned list (link, label, id) to CSV for use in step 2
data.to_csv("data/2_list_clean/clean_list.csv")